# 05 - Data analysis (PySpark DataFrame API)


In [ ]:
import os
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F


In [ ]:
spark_snowflake_pkg = os.getenv('SPARK_SNOWFLAKE_PACKAGE', 'net.snowflake:spark-snowflake_2.12:3.1.8')
snowflake_jdbc_pkg = os.getenv('SNOWFLAKE_JDBC_PACKAGE', 'net.snowflake:snowflake-jdbc:3.14.4')

spark = (
    SparkSession.builder
    .appName('PSet3 Data Analysis PySpark API')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.jars.packages', f"{spark_snowflake_pkg},{snowflake_jdbc_pkg}")
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)

print(spark.version)


In [ ]:
def get_env(name, default=None, required=False):
    value = os.getenv(name, default)
    if required and (value is None or str(value).strip() == ''):
        raise ValueError(f'Falta variable de entorno requerida: {name}')
    return value


def show_df(df, n=120):
    df.show(n, truncate=False)


In [ ]:
snowflake_host = get_env('SNOWFLAKE_HOST', '').strip()
snowflake_account = get_env('SNOWFLAKE_ACCOUNT', '').strip()
snowflake_port = get_env('SNOWFLAKE_PORT', '443').strip()

if not snowflake_host:
    if not snowflake_account:
        raise ValueError('Debes configurar SNOWFLAKE_HOST o SNOWFLAKE_ACCOUNT')
    snowflake_host = f"{snowflake_account}.snowflakecomputing.com"

if snowflake_port and snowflake_port != '443':
    snowflake_host = f"{snowflake_host}:{snowflake_port}"

sf_options = {
    'sfURL': snowflake_host,
    'sfUser': get_env('SNOWFLAKE_USER', required=True),
    'sfPassword': get_env('SNOWFLAKE_PASSWORD', required=True),
    'sfRole': get_env('SNOWFLAKE_ROLE', required=True),
    'sfWarehouse': get_env('SNOWFLAKE_WAREHOUSE', required=True),
    'sfDatabase': get_env('SNOWFLAKE_DATABASE', required=True),
    'sfSchema': get_env('SNOWFLAKE_SCHEMA_ANALYTICS', 'ANALYTICS'),
}

analytics_schema = get_env('SNOWFLAKE_SCHEMA_ANALYTICS', 'ANALYTICS')
obt_table_name = f"{analytics_schema}.OBT_TRIPS"

{k: ('***' if 'Password' in k else v) for k, v in sf_options.items()}


In [ ]:
raw_trips = (
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('dbtable', obt_table_name)
    .load()
)

trips = (
    raw_trips
    .filter(F.col('year').between(2015, 2025))
    .filter(F.col('pickup_datetime') >= F.to_timestamp(F.lit('2015-01-01 00:00:00')))
    .filter(F.col('pickup_datetime') < F.to_timestamp(F.lit('2026-01-01 00:00:00')))
    .filter(F.col('dropoff_datetime') >= F.to_timestamp(F.lit('2015-01-01 00:00:00')))
    .filter(F.col('dropoff_datetime') < F.to_timestamp(F.lit('2026-01-01 00:00:00')))
    .filter(F.abs(F.col('year') - F.col('source_year').cast('int')) <= 1)
    .filter(F.col('trip_distance').between(0.1, 100))
    .filter(F.col('trip_duration_min').between(1, 240))
    .filter(F.col('avg_speed_mph').isNull() | F.col('avg_speed_mph').between(1, 120))
    .filter(F.col('fare_amount').isNull() | (F.col('fare_amount') >= 0))
    .filter(F.col('total_amount').isNull() | (F.col('total_amount') >= 0))
    .filter(F.col('passenger_count').isNull() | F.col('passenger_count').between(1, 6))
    .cache()
)

trips.count()


## a) Top 10 zonas de pickup por volumen mensual


In [ ]:
pickup_rank_window = Window.partitionBy('year', 'month').orderBy(F.desc('trips'))

a_top_pickups = (
    trips
    .groupBy('year', 'month', 'pu_zone')
    .agg(F.count('*').alias('trips'))
    .withColumn('rn', F.row_number().over(pickup_rank_window))
    .filter(F.col('rn') <= 10)
    .select('year', 'month', 'pu_zone', 'trips')
    .orderBy('year', 'month', F.desc('trips'))
)

show_df(a_top_pickups)


## b) Top 10 zonas de dropoff por volumen mensual


In [ ]:
dropoff_rank_window = Window.partitionBy('year', 'month').orderBy(F.desc('trips'))

b_top_dropoffs = (
    trips
    .groupBy('year', 'month', 'do_zone')
    .agg(F.count('*').alias('trips'))
    .withColumn('rn', F.row_number().over(dropoff_rank_window))
    .filter(F.col('rn') <= 10)
    .select('year', 'month', 'do_zone', 'trips')
    .orderBy('year', 'month', F.desc('trips'))
)

show_df(b_top_dropoffs)


## c) Evolucion mensual de total_amount y tip_pct por borough


In [ ]:
c_amount_tip_by_borough = (
    trips
    .groupBy('year', 'month', 'pu_borough')
    .agg(
        F.round(F.sum('total_amount'), 2).alias('total_amount_sum'),
        F.round(F.avg('tip_pct'), 4).alias('avg_tip_pct')
    )
    .orderBy('year', 'month', 'pu_borough')
)

show_df(c_amount_tip_by_borough)


## d) Ticket promedio por service_type y mes


In [ ]:
d_avg_ticket = (
    trips
    .groupBy('year', 'month', 'service_type')
    .agg(F.round(F.avg('total_amount'), 2).alias('avg_ticket'))
    .orderBy('year', 'month', 'service_type')
)

show_df(d_avg_ticket)


## e) Viajes por hora del dia y dia de semana


In [ ]:
e_trips_by_hour_dow = (
    trips
    .groupBy('pickup_hour', 'day_of_week')
    .agg(F.count('*').alias('trips'))
    .orderBy(F.desc('trips'))
)

show_df(e_trips_by_hour_dow)


## f) p50/p90 de trip_duration_min por borough


In [ ]:
f_duration_percentiles = (
    trips
    .groupBy('pu_borough')
    .agg(
        F.percentile_approx('trip_duration_min', 0.5, 10000).alias('p50_duration'),
        F.percentile_approx('trip_duration_min', 0.9, 10000).alias('p90_duration')
    )
    .orderBy('pu_borough')
)

show_df(f_duration_percentiles)


## g) avg_speed_mph por franja horaria (6-9, 17-20) y borough


In [ ]:
g_speed_by_band = (
    trips
    .filter(F.col('pickup_hour').between(6, 9) | F.col('pickup_hour').between(17, 20))
    .withColumn(
        'time_band',
        F.when(F.col('pickup_hour').between(6, 9), F.lit('06-09'))
         .when(F.col('pickup_hour').between(17, 20), F.lit('17-20'))
         .otherwise(F.lit('other'))
    )
    .groupBy('time_band', 'pu_borough')
    .agg(F.round(F.avg('avg_speed_mph'), 2).alias('avg_speed_mph'))
    .orderBy('time_band', 'pu_borough')
)

show_df(g_speed_by_band)


## h) Participacion por payment_type_desc y relacion con tip_pct


In [ ]:
global_window = Window.partitionBy()

h_payment_mix = (
    trips
    .groupBy('payment_type_desc')
    .agg(
        F.count('*').alias('trips'),
        F.round(F.avg('tip_pct'), 4).alias('avg_tip_pct')
    )
    .withColumn('share_pct', F.round(F.col('trips') * 100.0 / F.sum('trips').over(global_window), 2))
    .select('payment_type_desc', 'trips', 'share_pct', 'avg_tip_pct')
    .orderBy(F.desc('trips'))
)

show_df(h_payment_mix)


## i) Rate codes con mayor trip_distance y total_amount


In [ ]:
i_rate_codes = (
    trips
    .groupBy('rate_code_desc')
    .agg(
        F.round(F.avg('trip_distance'), 2).alias('avg_trip_distance'),
        F.round(F.sum('total_amount'), 2).alias('sum_total_amount')
    )
    .orderBy(F.desc('sum_total_amount'), F.desc('avg_trip_distance'))
)

show_df(i_rate_codes)


## j) Mix yellow vs green por mes y borough


In [ ]:
j_service_mix = (
    trips
    .groupBy('year', 'month', 'pu_borough', 'service_type')
    .agg(F.count('*').alias('trips'))
    .orderBy('year', 'month', 'pu_borough', 'service_type')
)

show_df(j_service_mix, n=150)


## k) Top 20 flujos PU->DO por volumen y ticket promedio


In [ ]:
k_top_flows = (
    trips
    .groupBy('pu_zone', 'do_zone')
    .agg(
        F.count('*').alias('trips'),
        F.round(F.avg('total_amount'), 2).alias('avg_ticket')
    )
    .orderBy(F.desc('trips'))
    .limit(20)
)

show_df(k_top_flows)


## l) Distribucion de passenger_count y efecto en total_amount


In [ ]:
l_passenger_mix = (
    trips
    .groupBy('passenger_count')
    .agg(
        F.count('*').alias('trips'),
        F.round(F.avg('total_amount'), 2).alias('avg_total_amount')
    )
    .orderBy(F.desc('trips'))
)

show_df(l_passenger_mix)


## m) Impacto de tolls_amount y congestion_surcharge por zona


In [ ]:
m_surcharges_by_zone = (
    trips
    .groupBy('pu_zone')
    .agg(
        F.round(F.avg('tolls_amount'), 2).alias('avg_tolls'),
        F.round(F.avg('congestion_surcharge'), 2).alias('avg_congestion'),
        F.round(F.avg('total_amount'), 2).alias('avg_total_amount')
    )
    .orderBy(F.desc('avg_congestion'), F.desc('avg_tolls'))
    .limit(30)
)

show_df(m_surcharges_by_zone)


## n) Proporcion de viajes cortos vs largos por borough y estacionalidad


In [ ]:
n_short_vs_long = (
    trips
    .groupBy('year', 'month', 'pu_borough')
    .agg(
        F.sum(F.when(F.col('trip_distance') < 2, 1).otherwise(0)).alias('short_trips'),
        F.sum(F.when(F.col('trip_distance') >= 10, 1).otherwise(0)).alias('long_trips'),
        F.count('*').alias('total_trips')
    )
    .orderBy('year', 'month', 'pu_borough')
)

show_df(n_short_vs_long, n=150)


## o) Diferencias por vendor en avg_speed_mph y trip_duration_min


In [ ]:
o_vendor_diffs = (
    trips
    .groupBy('vendor_name')
    .agg(
        F.round(F.avg('avg_speed_mph'), 2).alias('avg_speed_mph'),
        F.round(F.avg('trip_duration_min'), 2).alias('avg_trip_duration_min'),
        F.count('*').alias('trips')
    )
    .orderBy(F.desc('trips'))
)

show_df(o_vendor_diffs)


## p) Relacion metodo de pago vs tip_amount por hora


In [ ]:
p_tip_by_hour_payment = (
    trips
    .groupBy('pickup_hour', 'payment_type_desc')
    .agg(
        F.round(F.avg('tip_amount'), 2).alias('avg_tip_amount'),
        F.round(F.avg('tip_pct'), 4).alias('avg_tip_pct'),
        F.count('*').alias('trips')
    )
    .orderBy('pickup_hour', F.desc('trips'))
)

show_df(p_tip_by_hour_payment, n=200)


## q) Zonas con percentil 99 de duracion/distancia fuera de rango


In [ ]:
q_zone_outliers = (
    trips
    .groupBy('pu_zone')
    .agg(
        F.percentile_approx('trip_duration_min', 0.99, 10000).alias('p99_duration'),
        F.percentile_approx('trip_distance', 0.99, 10000).alias('p99_distance')
    )
    .filter((F.col('p99_duration') > 180) | (F.col('p99_distance') > 30))
    .orderBy(F.desc('p99_duration'), F.desc('p99_distance'))
    .limit(50)
)

show_df(q_zone_outliers)


## r) Yield por milla (total_amount/trip_distance) por borough y hora


In [ ]:
r_yield_per_mile = (
    trips
    .withColumn(
        'yield_per_mile',
        F.when(F.col('trip_distance') > 0, F.col('total_amount') / F.col('trip_distance'))
    )
    .groupBy('pu_borough', 'pickup_hour')
    .agg(
        F.round(F.avg('yield_per_mile'), 2).alias('avg_yield_per_mile'),
        F.count('*').alias('trips')
    )
    .orderBy('pu_borough', 'pickup_hour')
)

show_df(r_yield_per_mile, n=200)


## s) Cambios YoY en volumen y ticket promedio por service_type


In [ ]:
yoy_window = Window.partitionBy('service_type').orderBy('year')

s_yoy = (
    trips
    .groupBy('year', 'service_type')
    .agg(
        F.count('*').alias('trips'),
        F.avg('total_amount').alias('avg_ticket')
    )
    .withColumn('trips_prev', F.lag('trips').over(yoy_window))
    .withColumn('avg_ticket_prev', F.lag('avg_ticket').over(yoy_window))
    .withColumn('avg_ticket', F.round(F.col('avg_ticket'), 2))
    .withColumn(
        'yoy_trips_pct',
        F.round(100.0 * (F.col('trips') - F.col('trips_prev')) / F.col('trips_prev'), 2)
    )
    .withColumn(
        'yoy_ticket_pct',
        F.round(100.0 * (F.col('avg_ticket') - F.col('avg_ticket_prev')) / F.col('avg_ticket_prev'), 2)
    )
    .select('year', 'service_type', 'trips', 'avg_ticket', 'yoy_trips_pct', 'yoy_ticket_pct')
    .orderBy('service_type', 'year')
)

show_df(s_yoy, n=50)


## t) Dias con alta congestion_surcharge vs dias normales


In [ ]:
t_by_day = (
    trips
    .groupBy('pickup_date')
    .agg(
        F.avg('congestion_surcharge').alias('avg_congestion'),
        F.avg('total_amount').alias('avg_total')
    )
)

congestion_p90 = t_by_day.agg(F.percentile_approx('avg_congestion', 0.9, 10000).alias('p90')).first()['p90']

t_congestion_days = (
    t_by_day
    .withColumn(
        'day_type',
        F.when(F.col('avg_congestion') >= F.lit(congestion_p90), F.lit('high_congestion_day'))
         .otherwise(F.lit('normal_day'))
    )
    .groupBy('day_type')
    .agg(
        F.count('*').alias('days'),
        F.round(F.avg('avg_congestion'), 3).alias('avg_congestion'),
        F.round(F.avg('avg_total'), 2).alias('avg_total_amount')
    )
    .orderBy('day_type')
)

show_df(t_congestion_days)


## Nota metodologica (lectura de negocio)

Para reporting ejecutivo, conviene complementar con una vista `business_clean` excluyendo categorias de baja interpretabilidad (`Unknown`, `N/A`, `EWR`) en borough/zone, sin reemplazar la respuesta oficial del PSet.


In [ ]:
business_clean = (
    trips
    .filter(~F.coalesce(F.col('pu_borough'), F.lit('Unknown')).isin('Unknown', 'N/A', 'EWR'))
    .filter(~F.coalesce(F.col('do_borough'), F.lit('Unknown')).isin('Unknown', 'N/A', 'EWR'))
    .filter(~F.coalesce(F.col('pu_zone'), F.lit('Unknown')).isin('Unknown', 'N/A'))
    .filter(~F.coalesce(F.col('do_zone'), F.lit('Unknown')).isin('Unknown', 'N/A'))
)

business_clean.select(F.count('*').alias('rows_business_clean')).show()
